In [1]:
import anthropic
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
import os


In [2]:
os.chdir('../classifications')


In [3]:
os.getcwd()

'/home/daniel/code/emestrada_parser/classifications'

In [5]:
df = pd.read_csv('quimica_statements.csv')
df.sample(5)

,subject,topic,year,exam,exercise,statement
63,química,Equilibrio Químico,2006,Reserva 3,Ejercicio 3 Opción A,Considérese el siguiente sistema en equilibrio...
447,química,Enlace Químico,2007,Reserva 2,Ejercicio 2 Opcion A,Explique:\na) Por qué el cloruro de hidrógeno ...
1320,química,Solubilidad,2019,Reserva 2,Ejercicio 3 Opción B,Se dispone de una disolución acuosa saturada d...
1203,química,Termoquímica,2006,Reserva 3,Ejercicio 5 Opción A,Dadas las ecuaciones termoquímicas siguientes:...
528,química,Enlace Químico,2022,Reserva 3,Ejercicio B3,a) El compuesto formado al enlazarse los eleme...


In [6]:
df["exam_name"] = df.apply(lambda row: f"{row["subject"]} - {row["topic"].lower()} - {row["year"]} - {row["exam"]} - {row["exercise"]}", axis=1)

Subset only the `solubilidad` topic

In [7]:
df = df[df.topic == "Solubilidad"]

In [8]:
df.sample(5)

,subject,topic,year,exam,exercise,statement,exam_name
1294,química,Solubilidad,2013,Reserva 1,Ejercicio 3 Opcion A,Escriba la ecuación que relaciona la solubilid...,química - solubilidad - 2013 - Reserva 1 - Eje...
1295,química,Solubilidad,2013,Reserva 2,Ejercicio 5 Opción B,"A 25*C el producto de solubilidad del MgF, es ...",química - solubilidad - 2013 - Reserva 2 - Eje...
1308,química,Solubilidad,2017,Junio,Ejercicio 5 Opción B,El producto de solubilidad del carbonato de ca...,química - solubilidad - 2017 - Junio - Ejercic...
1301,química,Solubilidad,2015,Junio,Ejercicio 3 Opción A,"Dada una disolución saturada de Mg(OH),, cuyo ...",química - solubilidad - 2015 - Junio - Ejercic...
1333,química,Solubilidad,2021,Reserva 2,Ejercicio C2,"La solubilidad del cromato de plata (Ag,CrO,) ...",química - solubilidad - 2021 - Reserva 2 - Eje...


In [9]:
df_to_ai = df[["exam_name", "statement"]]

In [10]:
df_to_ai.sample(10)

,exam_name,statement
1339,química - solubilidad - 2022 - Junio - Ejercic...,"El hidróxido de cobre(ID), Cu(OH),,es una sal ..."
1318,química - solubilidad - 2019 - Junio - Ejercic...,El PbCO; es una sal muy poco soluble en agua c...
1308,química - solubilidad - 2017 - Junio - Ejercic...,El producto de solubilidad del carbonato de ca...
1342,química - solubilidad - 2022 - Reserva 1 - Eje...,"A 25""%C, la constante de solubilidad del AgCI ..."
1303,química - solubilidad - 2015 - Reserva 4 - Eje...,"Sabiendo que el producto de solubilidad, K ,, ..."
1282,química - solubilidad - 2010 - Reserva 2 - Eje...,Los productos de solubilidad del cloruro de pl...
1325,química - solubilidad - 2020 - Reserva 2 - Eje...,"Sabiendo que el valor de K, del Mg(OH), a 25%C..."
1284,química - solubilidad - 2010 - Reserva 4 - Eje...,A 25%C el producto de solubilidad en agua del ...
1331,química - solubilidad - 2021 - Junio - Ejercic...,Una disolución saturada d yoduro de plomo(II) ...
1338,química - solubilidad - 2021 - Julio - Ejercic...,"A 25""C el producto de solubilidad del sulfuro ..."


In [11]:
df_to_ai.to_csv("solubilidad_statements.csv", sep="|", index = False)

In [20]:
load_dotenv('../etc/.env')
client = anthropic.Anthropic()

In [ ]:

def classify_exercises(topic: str, csv_file: str, instructions_md: str ) -> list[dict]:
    """
    enunciados: lista de dicts con keys año, convocatoria, ejercicio, texto
    retorna:    lista de dicts con keys año, convocatoria, ejercicio, tema, tipo_ejercicio
    """
    instructions = Path(instructions_md)

    # Carga el markdown con los criterios del tema
    system_prompt = instructions.read_text()

    # Carga el archivo csv
    csv = Path(csv_file)

    # create content of the prompt:
    content=f'''
    Eres un experto en exámenes de {topic} de Selectividad de la comunidad autónoma de Andalucía. En el archivo csv te adjunto dos columnas: exam_name y statement. Es un archivo que usa | como separador.

    En el archivo instructions_md (incluido en el system prompt) existen instrucciones para que te ayuden a clasificar cada tipo de ejercicio. Usa la clasificación que existe denominada como "tipo" en el archivo markdown.

    Aquí tienes un ejemplo para que sepas cómo parsear la primera columna del archivo csv fuente:
    química - equilibrio químico - 2000 - Junio - Ejercicio 3 Opción B --> subject - topic - year- exam - exercise

    El enunciado que tienes que clasificar se encuentra en la segunda columna (statement)

    Quiero que devuelvas un archivo csv, con las siguientes columnas:
    year, subject, topic, exam, exercise, exercise_type.

    Usa separador | para no tener problemas con las comas.

    Archivos fuente:
    csv: {csv.read_text()}
    '''
    # Formatea los enunciados como texto para el mensaje
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=2048,
        system=system_prompt,
        messages=[
            {"role": "user", "content": content}
        ]
    )

    # Parsear la respuesta CSV
    result = [i for i in response.content[0].text.strip().splitlines()]

    return result

In [36]:
response = classify_exercises(topic = "solubilidad", csv_file="solubilidad_statements.csv", instructions_md="solubilidad.md")

In [37]:
response

['```',
 'year|subject|topic|exam|exercise|exercise_type',
 '2010|química|solubilidad|Junio|Ejercicio 3 Opción A|cálculo de la solubilidad o el producto de solubilidad',
 '2010|química|solubilidad|Reserva 2|Ejercicio 3 Opcion A|efecto del ion común teórico',
 '2010|química|solubilidad|Reserva 3|Ejercicio 5 Opción B|cálculo de la solubilidad (efecto ion común)',
 '2010|química|solubilidad|Reserva 4|Ejercicio 6 Opción A|influencia del pH',
 '2011|química|solubilidad|Reserva 1|Ejercicio 3 Opción A|efecto del ion común teórico',
 '2011|química|solubilidad|Reserva 2|Ejercicio 3 Opción A|cuestiones teóricas',
 '2011|química|solubilidad|Reserva 4|Ejercicio 5 Opción B|cálculo de la solubilidad o el producto de solubilidad',
 '2011|química|solubilidad|Septiembre|Ejercicio 6 Opción A|cálculo de la solubilidad (efecto ion común)',
 '2012|química|solubilidad|Junio|Ejercicio 3 Opción A|efecto del ion común teórico',
 '2012|química|solubilidad|Reserva 3|Ejercicio 5 Opción B|cálculo de la solubilidad

In [38]:
response = response[1:]
response[:4]


['year|subject|topic|exam|exercise|exercise_type',
 '2010|química|solubilidad|Junio|Ejercicio 3 Opción A|cálculo de la solubilidad o el producto de solubilidad',
 '2010|química|solubilidad|Reserva 2|Ejercicio 3 Opcion A|efecto del ion común teórico',
 '2010|química|solubilidad|Reserva 3|Ejercicio 5 Opción B|cálculo de la solubilidad (efecto ion común)']

In [41]:
with open("response.csv", "w") as file:
    for line in response:
        file.write(f"{line}\n")

In [42]:
response

['year|subject|topic|exam|exercise|exercise_type',
 '2010|química|solubilidad|Junio|Ejercicio 3 Opción A|cálculo de la solubilidad o el producto de solubilidad',
 '2010|química|solubilidad|Reserva 2|Ejercicio 3 Opcion A|efecto del ion común teórico',
 '2010|química|solubilidad|Reserva 3|Ejercicio 5 Opción B|cálculo de la solubilidad (efecto ion común)',
 '2010|química|solubilidad|Reserva 4|Ejercicio 6 Opción A|influencia del pH',
 '2011|química|solubilidad|Reserva 1|Ejercicio 3 Opción A|efecto del ion común teórico',
 '2011|química|solubilidad|Reserva 2|Ejercicio 3 Opción A|cuestiones teóricas',
 '2011|química|solubilidad|Reserva 4|Ejercicio 5 Opción B|cálculo de la solubilidad o el producto de solubilidad',
 '2011|química|solubilidad|Septiembre|Ejercicio 6 Opción A|cálculo de la solubilidad (efecto ion común)',
 '2012|química|solubilidad|Junio|Ejercicio 3 Opción A|efecto del ion común teórico',
 '2012|química|solubilidad|Reserva 3|Ejercicio 5 Opción B|cálculo de la solubilidad (efecto

In [ ]:
df_response = pd.read_csv("response.csv", sep="|")

,year,subject,topic,exam,exercise,exercise_type
12,2013,química,solubilidad,Junio,Ejercicio 3 Opción A,cuestiones teóricas
53,2021,química,solubilidad,Reserva 3,Ejercicio B5,cuestiones teóricas
21,2015,química,solubilidad,Reserva 2,Ejercicio 3 Opcion A,cuestiones teóricas
19,2014,química,solubilidad,Septiembre,Ejercicio 3 Opción A,cuestiones teóricas
5,2011,química,solubilidad,Reserva 2,Ejercicio 3 Opción A,cuestiones teóricas
48,2020,química,solubilidad,Septiembre,B5,cuestiones teóricas
34,2018,química,solubilidad,Reserva 2,Ejercicio 3 Opción A,cuestiones teóricas
36,2018,química,solubilidad,Septiembre,Ejercicio 3 Opción B,cuestiones teóricas
38,2019,química,solubilidad,Reserva 1,Ejercicio 3 Opción A,cuestiones teóricas
31,2017,química,solubilidad,Septiembre,Ejercicio 3 Opción A,cuestiones teóricas


In [47]:
df_response.sort_values(["exercise_type", "year"])

,year,subject,topic,exam,exercise,exercise_type
5,2011,química,solubilidad,Reserva 2,Ejercicio 3 Opción A,cuestiones teóricas
12,2013,química,solubilidad,Junio,Ejercicio 3 Opción A,cuestiones teóricas
19,2014,química,solubilidad,Septiembre,Ejercicio 3 Opción A,cuestiones teóricas
21,2015,química,solubilidad,Reserva 2,Ejercicio 3 Opcion A,cuestiones teóricas
31,2017,química,solubilidad,Septiembre,Ejercicio 3 Opción A,cuestiones teóricas
34,2018,química,solubilidad,Reserva 2,Ejercicio 3 Opción A,cuestiones teóricas
36,2018,química,solubilidad,Septiembre,Ejercicio 3 Opción B,cuestiones teóricas
38,2019,química,solubilidad,Reserva 1,Ejercicio 3 Opción A,cuestiones teóricas
48,2020,química,solubilidad,Septiembre,B5,cuestiones teóricas
53,2021,química,solubilidad,Reserva 3,Ejercicio B5,cuestiones teóricas
